import fille 

In [25]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

Chargement des données

In [26]:
df = pd.read_csv("../data/processed/clean_data.csv")

In [27]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1547 entries, 0 to 1546
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   1547 non-null   int64  
 1   title                        1547 non-null   object 
 2   price                        1547 non-null   float64
 3   city_name                    1547 non-null   object 
 4   salon                        1547 non-null   float64
 5   nb_rooms                     1547 non-null   float64
 6   nb_baths                     1547 non-null   float64
 7   surface_area                 1547 non-null   float64
 8   Ascenseur                    1547 non-null   float64
 9   Balcon                       1547 non-null   float64
 10  Chauffage                    1547 non-null   float64
 11  Climatisation                1547 non-null   float64
 12  Concierge                    1547 non-null   float64
 13  Cuisine Équipée   

Séparation X / y

In [28]:
X = df.drop(columns=["price","title","city_name"])
y = df["price"]

Split train/test

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
# Vérification des shapes
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(1237, 21) (310, 21)
(1237,) (310,)


In [31]:
X.dtypes

Unnamed: 0                       int64
salon                          float64
nb_rooms                       float64
nb_baths                       float64
surface_area                   float64
Ascenseur                      float64
Balcon                         float64
Chauffage                      float64
Climatisation                  float64
Concierge                      float64
Cuisine Équipée                float64
Duplex                         float64
Meublé                         float64
Parking                        float64
Sécurité                       float64
Terrasse                       float64
city_name_encoded              float64
nb_appartements_ville            int64
prix_moyen_ville               float64
surface_moyenne_ville          float64
ecart_surface_moyenne_ville    float64
dtype: object

In [32]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "SVR": SVR(),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

trained_models = {}
training_times = {}


🔹 Entraînement des modèles

In [33]:
for name, model in models.items():
    try:
        # print(f"\n Modèle: {name}")
        start_time = time.time()
        model.fit(X_train, y_train)
        end_time = time.time()

        trained_models[name] = model
        training_times[name] = round(end_time - start_time, 2)

        # print(f"Entraîné en {training_times[name]} secondes")
    except Exception as e:
        print(f" Erreur lors de l'entraînement de {name}: {e}")

🔹 Évaluation des modèles

In [34]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Modèle": name,
        "MSE": round(mse, 2),
        "RMSE": round(rmse, 2),
        "MAE": round(mae, 2),
        "R2 Score": round(r2, 4),
        "Temps (s)": training_times[name]
    })

🔹 Résultats sous forme de DataFrame

In [35]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R2 Score", ascending=False)


🔹 Affichage


In [36]:
print("\n Résultats des modèles :")
display(results_df)


 Résultats des modèles :


,Modèle,MSE,RMSE,MAE,R2 Score,Temps (s)
1,Random Forest,0.02,0.14,0.09,0.4758,0.36
3,Gradient Boosting,0.02,0.14,0.09,0.4515,0.29
0,Linear Regression,0.02,0.15,0.10,0.4000,0.01
2,SVR,0.03,0.18,0.14,0.0590,0.06


Validation croisée

In [37]:
for name, model in trained_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    print(f"{name} | Cross-val R2: {scores.mean():.4f} ± {scores.std():.4f}")

Linear Regression | Cross-val R2: 0.3881 ± 0.0696
Random Forest | Cross-val R2: 0.4196 ± 0.0476
SVR | Cross-val R2: -0.0009 ± 0.0194
Gradient Boosting | Cross-val R2: 0.4039 ± 0.0662


In [38]:
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0]
}

In [39]:
gbr = GradientBoostingRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=gbr,
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,  # Utilise tous les cœurs dispo
    verbose=2   # Affiche la progression
)

grid_search.fit(X_train, y_train)
print("🔍 Meilleurs hyperparamètres trouvés :")
print(grid_search.best_params_)

print("\n🏆 Meilleur score R² (cross-val) :")
print(round(grid_search.best_score_, 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
🔍 Meilleurs hyperparamètres trouvés :
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

🏆 Meilleur score R² (cross-val) :
0.43


In [43]:
joblib.dump(grid_search.best_estimator_, "../models/model.pkl")

['../models/model.pkl']